# 🏥 Medical Image Captioning with VGG16 + LSTM\n\nThis notebook implements an end-to-end captioning pipeline on the **ROCO radiology dataset**:\n\n1. Feature extraction using **VGG16** (pretrained on ImageNet)\n2. Caption preprocessing and tokenization\n3. **LSTM** decoder training\n4. **BLEU** score evaluation\n5. Caption generation demo\n\n[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ankit-thepe/MedCaption-LSTM/blob/main/notebooks/Medical_Image_Captioning_ROCO.ipynb)

## 1. Install Dependencies

In [ ]:
!pip install -q kaggle tensorflow scikit-learn nltk tqdm Pillow

## 2. Download ROCO Dataset

In [ ]:
# Upload kaggle.json first: Runtime > Upload files\nimport os\nos.makedirs('/root/.kaggle', exist_ok=True)\n!cp kaggle.json /root/.kaggle/\n!chmod 600 /root/.kaggle/kaggle.json\n\n!kaggle datasets download -d virajbagal/roco-dataset\n!unzip -q roco-dataset.zip -d /content/roco

## 3. Configuration

In [ ]:
import os\nimport pickle\nimport re\nimport csv\nimport numpy as np\nfrom tqdm.notebook import tqdm\n\nBASE_DIR    = '/content/roco/all_data/train/radiology'\nIMAGE_DIR   = os.path.join(BASE_DIR, 'images')\nCAPTION_CSV = os.path.join(BASE_DIR, 'traindata.csv')\nWORK_DIR    = '/content'\n\n# Model hyperparameters\nEMBED_DIM  = 256\nLSTM_UNITS = 256\nDROPOUT    = 0.4\nEPOCHS     = 20\nBATCH_SIZE = 32

## 4. Extract VGG16 Image Features

In [ ]:
from tensorflow.keras.applications.vgg16 import VGG16, preprocess_input\nfrom tensorflow.keras.preprocessing.image import load_img, img_to_array\nfrom tensorflow.keras.models import Model\n\n# Load VGG16 — remove top classification layer to get 4096-dim features\nvgg = VGG16()\nextractor = Model(inputs=vgg.inputs, outputs=vgg.layers[-2].output)\nextractor.trainable = False\nprint(extractor.summary())

In [ ]:
features = {}\nfor fname in tqdm(os.listdir(IMAGE_DIR)[:200]):   # Use more for full training\n    if not fname.lower().endswith(('.jpg','.jpeg','.png')):\n        continue\n    img  = load_img(os.path.join(IMAGE_DIR, fname), target_size=(224, 224))\n    arr  = img_to_array(img).reshape((1, 224, 224, 3))\n    feat = extractor.predict(preprocess_input(arr), verbose=0)\n    features[os.path.splitext(fname)[0]] = feat\n\npickle.dump(features, open(os.path.join(WORK_DIR, 'features.pkl'), 'wb'))\nprint(f'Extracted features for {len(features)} images')

## 5. Load & Preprocess Captions

In [ ]:
def load_captions(csv_path):\n    mapping = {}\n    with open(csv_path, 'r') as f:\n        for row in csv.DictReader(f):\n            img_id = os.path.splitext(row['name'])[0]\n            if img_id in features:\n                mapping.setdefault(img_id, []).append(row['caption'])\n    return mapping\n\ndef clean_caption(c):\n    c = re.sub(r'[^a-z\\s]', '', c.lower())\n    c = re.sub(r'\\s+', ' ', c).strip()\n    return 'startseq ' + ' '.join(w for w in c.split() if len(w)>1) + ' endseq'\n\nmapping = load_captions(CAPTION_CSV)\nfor k in mapping:\n    mapping[k] = [clean_caption(c) for c in mapping[k]]\n\nall_captions = [c for caps in mapping.values() for c in caps]\nprint(f'Images: {len(mapping)} | Captions: {len(all_captions)}')

## 6. Tokenize & Compute Vocabulary

In [ ]:
from tensorflow.keras.preprocessing.text import Tokenizer\n\ntokenizer = Tokenizer()\ntokenizer.fit_on_texts(all_captions)\nvocab_size = len(tokenizer.word_index) + 1\nmax_length = max(len(c.split()) for c in all_captions)\n\npickle.dump(tokenizer, open(os.path.join(WORK_DIR, 'tokenizer.pkl'), 'wb'))\nprint(f'Vocabulary size : {vocab_size}')\nprint(f'Max caption len : {max_length}')

## 7. Build VGG16 + LSTM Model

In [ ]:
from tensorflow.keras.layers import Input, Dense, LSTM, Embedding, Dropout, add\nfrom tensorflow.keras.utils import plot_model\n\n# Image branch\nimg_in   = Input(shape=(4096,), name='image_features')\nimg_drop = Dropout(DROPOUT)(img_in)\nimg_den  = Dense(EMBED_DIM, activation='relu')(img_drop)\n\n# Text branch\nseq_in   = Input(shape=(max_length,), name='sequence_input')\nseq_emb  = Embedding(vocab_size, EMBED_DIM, mask_zero=True)(seq_in)\nseq_drop = Dropout(DROPOUT)(seq_emb)\nseq_lstm = LSTM(LSTM_UNITS)(seq_drop)\n\n# Decoder\nmerged   = add([img_den, seq_lstm])\ndec      = Dense(EMBED_DIM, activation='relu')(merged)\noutput   = Dense(vocab_size, activation='softmax')(dec)\n\nmodel = Model(inputs=[img_in, seq_in], outputs=output, name='MedCaption_LSTM')\nmodel.compile(loss='categorical_crossentropy', optimizer='adam')\nmodel.summary()\nplot_model(model, show_shapes=True)

## 8. Train the Model

In [ ]:
import tensorflow as tf\nfrom sklearn.model_selection import train_test_split\nfrom tensorflow.keras.preprocessing.sequence import pad_sequences\nfrom tensorflow.keras.utils import to_categorical\nfrom tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint\n\ndef data_generator(keys, mapping, features, tokenizer, max_length, vocab_size, bs):\n    X1, X2, y = [], [], []\n    n = 0\n    while True:\n        for key in keys:\n            for cap in mapping[key]:\n                seq = tokenizer.texts_to_sequences([cap])[0]\n                for i in range(1, len(seq)):\n                    X1.append(features[key][0])\n                    X2.append(pad_sequences([seq[:i]], maxlen=max_length)[0])\n                    y.append(to_categorical([seq[i]], num_classes=vocab_size)[0])\n            n += 1\n            if n == bs and len(X1):\n                yield (tf.constant(np.array(X1), tf.float32),\n                       tf.constant(np.array(X2), tf.float32)), tf.constant(np.array(y), tf.float32)\n                X1, X2, y = [], [], []\n                n = 0\n\nkeys = list(mapping.keys())\ntrain_keys, val_keys = train_test_split(keys, test_size=0.2, random_state=42)\n\nmodel.fit(\n    data_generator(train_keys, mapping, features, tokenizer, max_length, vocab_size, BATCH_SIZE),\n    epochs=EPOCHS,\n    steps_per_epoch=max(1, len(train_keys) // BATCH_SIZE),\n    callbacks=[\n        ModelCheckpoint('/content/best_model.keras', save_best_only=True, verbose=1),\n        EarlyStopping(patience=3, verbose=1)\n    ],\n    verbose=1\n)

## 9. Evaluate with BLEU Score

In [ ]:
import nltk\nnltk.download('punkt')\nfrom nltk.translate.bleu_score import corpus_bleu\nimport keras\n\ndef idx_to_word(idx, tokenizer):\n    return next((w for w, i in tokenizer.word_index.items() if i == idx), None)\n\ndef predict_caption(model, feat, tokenizer, max_length):\n    in_text = 'startseq'\n    for _ in range(max_length):\n        seq  = pad_sequences([tokenizer.texts_to_sequences([in_text])[0]], maxlen=max_length)\n        word = idx_to_word(int(np.argmax(model.predict([feat, seq], verbose=0))), tokenizer)\n        if not word or word == 'endseq': break\n        in_text += ' ' + word\n    return in_text.replace('startseq','').strip()\n\nbest_model = keras.models.load_model('/content/best_model.keras')\nactual, predicted = [], []\nfor key in tqdm(val_keys[:50]):   # Use all val_keys for full evaluation\n    actual.append([c.split() for c in mapping[key]])\n    predicted.append(predict_caption(best_model, features[key], tokenizer, max_length).split())\n\nprint(f'BLEU-1: {corpus_bleu(actual, predicted, weights=(1,0,0,0)):.4f}')\nprint(f'BLEU-2: {corpus_bleu(actual, predicted, weights=(.5,.5,0,0)):.4f}')

## 10. Generate Sample Captions

In [ ]:
from PIL import Image\nimport matplotlib.pyplot as plt\n\ndef show_caption(image_id):\n    img_path = os.path.join(IMAGE_DIR, image_id + '.jpg')\n    image    = Image.open(img_path)\n    caption  = predict_caption(best_model, features[image_id], tokenizer, max_length)\n\n    fig, ax = plt.subplots(1, 1, figsize=(8, 6))\n    ax.imshow(image, cmap='gray')\n    ax.set_title(f'Caption: {caption}', fontsize=11, wrap=True, pad=12)\n    ax.axis('off')\n    plt.tight_layout()\n    plt.show()\n    print(f'Ground truth: {mapping[image_id][0]}')\n\n# Test on a few images\nfor img_id in list(mapping.keys())[:3]:\n    show_caption(img_id)